# QAssistant Jupyter Notebook 复现实验

这个 notebook 演示如何在 Jupyter 中直接调用项目代码完成以下流程：

1. 初始化项目路径。
2. 检查环境变量和配置文件。
3. 调用 CLI 对应的内部工具列出产物。
4. 使用本地 PDF 研报复现链路生成策略样例代码。
5. 可选运行完整研报生成流水线。

注意：真实 LLM 调用需要把 `.env` 中的 `MY_*` 占位值替换为你自己的模型配置。

In [ ]:
from pathlib import Path
import json
import os
import sys

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "src" / "cli.py").exists() and (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Cannot locate QAssistant project root. Please run this notebook inside the project tree.")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

## 1. 加载环境变量并检查是否可调用真实 LLM

In [ ]:
try:
    from dotenv import load_dotenv
    load_dotenv(PROJECT_ROOT / ".env")
except Exception as exc:
    print(f"python-dotenv unavailable or failed to load .env: {exc}")

REQUIRED_LLM_ENV = [
    "DS_MODEL_NAME", "DS_BASE_URL", "DS_API_KEY",
    "VLM_MODEL_NAME", "VLM_BASE_URL", "VLM_API_KEY",
    "EMBEDDING_MODEL_NAME", "EMBEDDING_BASE_URL", "EMBEDDING_API_KEY",
]

env_status = {}
for key in REQUIRED_LLM_ENV:
    value = os.getenv(key, "")
    env_status[key] = {
        "set": bool(value),
        "placeholder": value.startswith("MY_") or value == "",
    }

READY_FOR_LLM = all(item["set"] and not item["placeholder"] for item in env_status.values())
print(json.dumps(env_status, ensure_ascii=False, indent=2))
print(f"READY_FOR_LLM = {READY_FOR_LLM}")

if not READY_FOR_LLM:
    print("当前环境变量仍为空或为 MY_* 占位值。真实 LLM 单元会自动跳过。")

## 2. 校验配置文件

In [ ]:
from src.pipeline.config_utils import validate_config_file

CONFIG_PATH = PROJECT_ROOT / "my_config.yaml"
validation = validate_config_file(CONFIG_PATH, strict_env=False)
print(json.dumps(validation, ensure_ascii=False, indent=2))

## 3. 扫描当前输出产物

In [ ]:
from src.pipeline.config_utils import list_output_artifacts

artifacts = list_output_artifacts(CONFIG_PATH)
print(json.dumps(artifacts, ensure_ascii=False, indent=2))

## 4. CLI help smoke test

这个单元不会调用 LLM，只验证当前环境能加载 QAssistant CLI。

In [ ]:
import subprocess

completed = subprocess.run(
    [sys.executable, "-m", "src.cli", "--help"],
    cwd=PROJECT_ROOT,
    text=True,
    capture_output=True,
    check=False,
)
print(completed.stdout)
if completed.returncode != 0:
    print(completed.stderr)
assert completed.returncode == 0

## 5. PDF 研报复现：生成策略样例代码

该流程会调用真实 LLM。若 `.env` 仍是 `MY_*` 占位值，本单元会跳过。默认使用 `assets/example_reports/PopMart.pdf`，并限制解析前 3 页用于快速试跑。

In [ ]:
PDF_PATH = PROJECT_ROOT / "assets" / "example_reports" / "PopMart.pdf"
REPORT_ID = "notebook_popmart_sample"
MAX_PAGES = 3

if not PDF_PATH.exists():
    print(f"PDF not found: {PDF_PATH}")
elif not READY_FOR_LLM:
    print("Skip reproduction run because READY_FOR_LLM is False.")
else:
    from src.pipeline.reproduction_runner import run_reproduction_pipeline

    reproduction_result = await run_reproduction_pipeline(
        pdf_path=str(PDF_PATH),
        config_file_path=str(CONFIG_PATH),
        report_id=REPORT_ID,
        max_pages=MAX_PAGES,
    )
    print(json.dumps(reproduction_result["summary"], ensure_ascii=False, indent=2))

## 6. 查看研报复现产物

In [ ]:
from src.pipeline.config_utils import resolve_working_dir

working_dir = resolve_working_dir(CONFIG_PATH)
reproduction_dir = working_dir / "report_reproduction" / REPORT_ID
print(f"Expected reproduction dir: {reproduction_dir}")

if reproduction_dir.exists():
    for path in sorted(reproduction_dir.iterdir()):
        print(path.name)
    strategy_path = reproduction_dir / "sample_strategy.py"
    if strategy_path.exists():
        print("\n--- sample_strategy.py preview ---")
        print(strategy_path.read_text(encoding="utf-8")[:3000])
else:
    print("产物目录尚未生成。请先配置真实 LLM 环境变量并运行上一个单元。")

## 7. 可选：运行完整研报生成流水线

完整流水线会调用真实 LLM、搜索和数据工具，耗时较长。默认 `RUN_FULL_REPORT = False`。

In [ ]:
RUN_FULL_REPORT = False

if not RUN_FULL_REPORT:
    print("Skip full report pipeline. Set RUN_FULL_REPORT = True to run it.")
elif not READY_FOR_LLM:
    print("Skip full report pipeline because READY_FOR_LLM is False.")
else:
    from src.pipeline.report_runner import run_report_pipeline

    report_result = await run_report_pipeline(
        config_file_path=str(CONFIG_PATH),
        resume=True,
        max_concurrent=1,
        auto_generate_tasks=False,
        collect_max_iterations=5,
        analysis_max_iterations=5,
        report_max_iterations=5,
        echo=False,
    )
    print(json.dumps(report_result, ensure_ascii=False, indent=2))